In [1]:
import os
import numpy as np
import SimpleITK as sitk
from tqdm import tqdm
import json
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap
from scipy.ndimage import binary_dilation

# PROSTATEx Preprocessing
Resample all MRI modalities (T2, B50, B400, B800, HBV, ADC) to a common spacing, register them to the prostate zone ROI, crop around the prostate region, and export per-slice 2D NIfTI files with distortion pivot metadata.

In [2]:
def resample_sitk_image(image, new_spacing=(0.5, 0.5, 3), is_roi=False):
    """
    Resample a SimpleITK image to a new spacing.
    Uses nearest-neighbor interpolation for ROIs and linear interpolation otherwise.

    Parameters:
    - image: SimpleITK.Image, the input image to resample.
    - new_spacing: tuple of floats, target spacing (mm) in (x, y, z).
    - is_roi: bool, if True use nearest-neighbor to preserve label values.

    Returns:
    - SimpleITK.Image, the resampled image.
    """
    original_size = image.GetSize()
    original_spacing = image.GetSpacing()

    # Compute new grid size to preserve physical extent
    new_size = [
        int(round(original_size[0] * (original_spacing[0] / new_spacing[0]))),
        int(round(original_size[1] * (original_spacing[1] / new_spacing[1]))),
        int(round(original_size[2] * (original_spacing[2] / new_spacing[2])))
    ]

    interpolation_method = sitk.sitkNearestNeighbor if is_roi else sitk.sitkLinear

    resampler = sitk.ResampleImageFilter()
    resampler.SetSize(new_size)
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetInterpolator(interpolation_method)
    resampler.SetOutputOrigin(image.GetOrigin())
    resampler.SetOutputDirection(image.GetDirection())

    return resampler.Execute(image)

def resample_to_reference(input_image, reference_image):
    """
    Resample input_image into the physical space of reference_image
    (matching size, spacing, origin, and direction).
    """
    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(reference_image)
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetTransform(sitk.AffineTransform(input_image.GetDimension()))
    resampler.SetOutputSpacing(reference_image.GetSpacing())
    resampler.SetSize(reference_image.GetSize())
    resampler.SetOutputDirection(reference_image.GetDirection())
    resampler.SetOutputOrigin(reference_image.GetOrigin())
    resampler.SetDefaultPixelValue(0)

    return resampler.Execute(input_image)

def numpy_to_python(value):
    """Convert numpy types to native Python types for JSON serialization."""
    if isinstance(value, np.integer):
        return int(value)
    elif isinstance(value, np.floating):
        return float(value)
    elif isinstance(value, np.ndarray):
        return value.tolist()
    else:
        return value

In [3]:
root = '/workspace/data/'
data_root = os.path.join(root, 'nifti')
save_root = os.path.join(root, 'preprocessed')
pztz_root = os.path.join(root, 'pztz')

pats = os.listdir(data_root)
print(f"Found {len(pats)} patients in the dataset.")

Found 5 patients in the dataset.


In [4]:
# Create output directories for each modality
for modality in ["PZTZ", "T2", "B50", "B400", "B800", "HBV", "ADC"]:
    os.makedirs(os.path.join(save_root, modality), exist_ok=True)

In [5]:
distortion_pivot = dict()
crop_coor = dict()
error_img = dict()

error_cnt = dict()
for pat in pats:
    error_cnt[pat] = 0

for pat in tqdm(pats):

    # --- Load all modality files, skip patient if any required file is missing ---
    roi_path = os.path.join(pztz_root, pat + ".nii.gz")

    t2_path = os.path.join(data_root, pat, 'T2.nii.gz')
    if not os.path.isfile(t2_path):
        print("Missing T2:", t2_path)
        error_img[t2_path] = 'no T2 file'
        error_cnt[pat] += 1
        continue

    # B50 fallback: try B100 if B50 doesn't exist
    b50_path = os.path.join(data_root, pat, 'B50.nii.gz')
    if not os.path.isfile(b50_path):
        b50_path = os.path.join(data_root, pat, 'B100.nii.gz')
    if not os.path.isfile(b50_path):
        print("Missing B50:", b50_path)
        error_img[b50_path] = 'no B50 file'
        error_cnt[pat] += 1
        continue

    # B400 fallback: try B500 if B400 doesn't exist
    b400_path = os.path.join(data_root, pat, 'B400.nii.gz')
    if not os.path.isfile(b400_path):
        b400_path = os.path.join(data_root, pat, 'B500.nii.gz')
    if not os.path.isfile(b400_path):
        print("Missing B400:", b400_path)
        error_img[b400_path] = 'no B400 file'
        error_cnt[pat] += 1
        continue

    b800_path = os.path.join(data_root, pat, 'B800.nii.gz')
    if not os.path.isfile(b800_path):
        print("Missing B800:", b800_path)
        error_img[b800_path] = 'no B800 file'
        error_cnt[pat] += 1
        continue

    # HBV fallback: try B1400 if HBV doesn't exist
    HBV_path = os.path.join(data_root, pat, 'HBV.nii.gz')
    if not os.path.isfile(HBV_path):
        HBV_path = os.path.join(data_root, pat, 'B1400.nii.gz')
    if not os.path.isfile(HBV_path):
        print("Missing HBV:", HBV_path)
        error_img[HBV_path] = 'no HBV file'
        error_cnt[pat] += 1
        continue

    ADC_path = os.path.join(data_root, pat, 'ADC.nii.gz')
    if not os.path.isfile(ADC_path):
        print("Missing ADC:", ADC_path)
        error_img[ADC_path] = 'no ADC file'
        error_cnt[pat] += 1
        continue

    # --- Read images ---
    roi_sitk = sitk.ReadImage(roi_path)
    t2_sitk = sitk.ReadImage(t2_path)
    b50_sitk = sitk.ReadImage(b50_path)
    b400_sitk = sitk.ReadImage(b400_path)
    b800_sitk = sitk.ReadImage(b800_path)
    HBV_sitk = sitk.ReadImage(HBV_path)
    ADC_sitk = sitk.ReadImage(ADC_path)

    # --- Resample ROI and T2 to uniform spacing, then register diffusion maps to ROI space ---
    roi_sitk = resample_sitk_image(roi_sitk, new_spacing=(0.5, 0.5, 3), is_roi=True)
    t2_sitk = resample_sitk_image(t2_sitk, new_spacing=(0.5, 0.5, 3), is_roi=False)

    b50_sitk = resample_to_reference(b50_sitk, roi_sitk)
    b400_sitk = resample_to_reference(b400_sitk, roi_sitk)
    b800_sitk = resample_to_reference(b800_sitk, roi_sitk)
    HBV_sitk = resample_to_reference(HBV_sitk, roi_sitk)
    ADC_sitk = resample_to_reference(ADC_sitk, roi_sitk)

    # --- Convert to numpy arrays ---
    t2_arr = sitk.GetArrayFromImage(t2_sitk)
    b50_arr = sitk.GetArrayFromImage(b50_sitk)
    b400_arr = sitk.GetArrayFromImage(b400_sitk)
    b800_arr = sitk.GetArrayFromImage(b800_sitk)
    HBV_arr = sitk.GetArrayFromImage(HBV_sitk)
    ADC_arr = sitk.GetArrayFromImage(ADC_sitk)

    # --- Find prostate bounding box from ROI nonzero region ---
    roi_arr = sitk.GetArrayFromImage(roi_sitk)
    nonzero = roi_arr.nonzero()
    if len(nonzero[0]) == 0:
        continue

    z_min, z_max = nonzero[0].min(), nonzero[0].max()
    x_min, x_max = nonzero[2].min(), nonzero[2].max()
    y_min, y_max = nonzero[1].min(), nonzero[1].max()

    # Expand x/y crop region to 2x the prostate extent, keep z tight
    x_size = (x_max - x_min) * 2
    y_size = (y_max - y_min) * 2
    z_size = z_max - z_min + 1

    # Center the expanded crop around the prostate
    x_start = max(0, x_min - (x_size // 4))
    y_start = max(0, y_min - (y_size // 4))
    z_start = z_min

    # Clamp to image bounds
    x_size = min(x_size, roi_sitk.GetSize()[0] - x_start)
    y_size = min(y_size, roi_sitk.GetSize()[1] - y_start)
    z_size = min(z_size, roi_sitk.GetSize()[2] - z_start)

    x_size, y_size, z_size = int(x_size), int(y_size), int(z_size)
    x_start, y_start, z_start = int(x_start), int(y_start), int(z_start)

    # Trim top/bottom z slices (boundary artifacts)
    z_size -= 2
    z_start += 1

    crop_coor[pat] = dict()
    crop_coor[pat]['x_start'] = x_start
    crop_coor[pat]['x_size'] = x_size
    crop_coor[pat]['y_start'] = y_start
    crop_coor[pat]['y_size'] = y_size
    crop_coor[pat]['z_start'] = z_start
    crop_coor[pat]['z_size'] = z_size

    # --- Extract per-slice 2D images and save ---
    for i in range(z_start, z_start + z_size):
        roi = roi_arr[i]
        t2 = t2_arr[i]
        b50 = b50_arr[i]
        b400 = b400_arr[i]
        b800 = b800_arr[i]
        HBV = HBV_arr[i]
        ADC = ADC_arr[i]

        # Compute prostate centroid and bounds for distortion pivot points
        prostate = np.where(roi != 0)
        if len(prostate[0]) == 0:
            continue

        prostate_y_min, prostate_y_max = prostate[0].min(), prostate[0].max()
        prostate_y_median = int(round((prostate_y_min + prostate_y_max) / 2))
        prostate_x_min, prostate_x_max = prostate[1].min(), prostate[1].max()
        prostate_x_median = int(round((prostate_x_min + prostate_x_max) / 2))

        y_end = y_start + y_size
        y_median = int(round((prostate_y_median + y_end) / 2))

        # Store 3x3 grid of pivot points (prostate center/edge x crop boundary)
        slice_name = pat + "_" + str(i)
        distortion_pivot[slice_name] = [
            (prostate_y_median, prostate_x_min),
            (prostate_y_median, prostate_x_median),
            (prostate_y_median, prostate_x_max),
            (y_median, prostate_x_min),
            (y_median, prostate_x_median),
            (y_median, prostate_x_max),
            (y_end, prostate_x_min),
            (y_end, prostate_x_median),
            (y_end, prostate_x_max),
        ]

        # Skip outlier slices (constant or near-constant B50 signal)
        if b50.min() == b50.max() or b50.std() < 0.001:
            print("Outlier B50 slice:", slice_name)
            error_img[slice_name] = 'outlier'
            error_cnt[pat] += 1
            continue

        # Save each modality as a 2D NIfTI slice
        for modality, arr in [("PZTZ", roi), ("T2", t2), ("B50", b50),
                              ("B400", b400), ("B800", b800), ("HBV", HBV), ("ADC", ADC)]:
            save_path = os.path.join(save_root, modality, slice_name + ".nii.gz")
            sitk.WriteImage(sitk.GetImageFromArray(arr), save_path)

    # --- Save metadata JSONs (updated after each patient) ---
    for fname, data in [('distortion_pivot.json', distortion_pivot),
                        ('crop_coor.json', crop_coor),
                        ('error_img_list.json', error_img),
                        ('error_cnt.json', error_cnt)]:
        with open(os.path.join(save_root, fname), 'w') as f:
            json.dump(data, f, indent=4, default=numpy_to_python)

100%|██████████| 5/5 [00:05<00:00,  1.20s/it]
